# Stage 00b — Figure Matching

**What this notebook does:**  
Reads the *Brief Description of the Drawings* text from the PatSeer Excel and
assigns a figure label to every downloaded image using **pure positional
matching** — no OCR, no external models.

The logic is:
1. Load all files for a patent from `raw/{patent_id}/` in order: `img*` → `D*` → `FAT*`
2. For `D` and `FAT` files, detect horizontal/vertical whitespace bands and split
   pages that contain multiple figures
3. Build one global ordered crop list
4. Parse the description text → ordered figure key list (FIG. 1, FIG. 2A, …)
5. If **crop count == description count** → positional 1-to-1 match → `_F` names
6. If **counts differ** → flag everything `_Fu` (needs human review via Stage 01)

**Output naming:**

| Before | After | Meaning |
|--------|-------|---------|
| `US…_img003.png` | `US…_img003_F002B.png` | matched to FIG. 2B |
| `US…_img003.png` | `US…_img003_Fu001.png` | count mismatch — needs review |
| `US…_D00005.png` | `US…_D00005_crop01_F001.png` | first crop of a split D sheet |
| `US…_D00005.png` | kept as-is | original D file always preserved |

**Where it fits in the pipeline:**
```
00a  (PatSeer download)
 ↓
00b  ←  YOU ARE HERE  (positional matching → _F / _Fu labels)
 ↓
01   (optional OCR fallback for _Fu files that need review)
 ↓
02   (pad + resize to 518×518)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and Excel index (reads `description_of_drawings` column) |
| 2 | Dry-run on one patent — shows split detection, crop list, rename preview; **no files changed** |
| 3 | Full run — match and rename all patents in `raw/` |
| 4 | Print matching report (_F count, _Fu count, originals kept) |

In [1]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

# Load config_loader from an explicit absolute path — immune to sys.path ordering
# and sys.modules cache (the two bugs that keep breaking the plain import).
_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod        # register before exec so relative imports inside it work
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

print(f"config_loader loaded from: {_cl_path}")

# sys.path for the remaining imports (figure_matcher, extractor, etc.)
for p in [str(repo_root), str(src_dir)]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(src_dir))

from figure_matcher import (
    parse_description_figures,
    detect_split_needed,
    match_positionally,
    rename_matched_files,
)
from extractor import load_patseer_excel

cfg         = load_config()
excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
print(f"Loaded {len(excel_index)} patents from Excel.")

config_loader loaded from: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/config_loader.py


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1635__dataset_26_06_02.xlsx  (1635 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [2]:
# ─── Dry-run on ONE patent — inspect results before committing any renames ────
TEST_PATENT = "US2015014475A1"   # change to any patent ID present in raw/

raw_dir         = cfg["paths"]["raw_images"]
img_dir         = raw_dir / TEST_PATENT
meta            = excel_index.get(TEST_PATENT, {})
description_text = meta.get("description_of_drawings") or ""

print(f"Patent      : {TEST_PATENT}")
print(f"Img dir     : {img_dir}")
print(f"Dir exists  : {img_dir.exists()}")
print(f"Desc length : {len(description_text)} chars")
print()

# Parse description
fig_keys = parse_description_figures(description_text)
print(f"Description figures ({len(fig_keys)}): {fig_keys}")
print()

# File inventory
import re
img_files  = sorted(img_dir.glob(f"{TEST_PATENT}_img*.png"))
d_files    = sorted(img_dir.glob(f"{TEST_PATENT}_D*.png"))
fat_files  = sorted(img_dir.glob(f"{TEST_PATENT}_FAT*.png"))
print(f"img files   : {len(img_files)}")
print(f"D files     : {len(d_files)}")
print(f"FAT files   : {len(fat_files)}")
print()

# Split detection for D and FAT
for f in d_files + fat_files:
    n = detect_split_needed(f)
    print(f"  {f.name}  →  {n} sub-figure(s)")
if d_files or fat_files:
    print()

# Run matching (does NOT rename anything)
matches = match_positionally(TEST_PATENT, img_dir, description_text)

total_crops = len(matches)
total_figs  = len(fig_keys)
count_ok    = total_crops == total_figs

print(f"Global crops  : {total_crops}")
print(f"Desc figures  : {total_figs}")
print(f"Count match   : {'✓ EXACT — positional match' if count_ok else '✗ MISMATCH — all will be _Fu'}")
print()

# Preview rename table
import pandas as pd
from IPython.display import display

df_preview = pd.DataFrame([
    {
        "source_file":     m["source_file"],
        "type":            m["source_type"],
        "split":           m["was_split"],
        "fig_key":         m["fig_key"],
        "needs_review":    m["needs_review"],
        "output_filename": m["output_filename"],
    }
    for m in matches
])
display(df_preview)

Patent      : US2015014475A1
Img dir     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Processed Images DataSets & Model Outputs/1635/raw/US2015014475A1
Dir exists  : False
Desc length : 2605 chars

Description figures (12): ['1', '2A', '3A', '3B', '3C', '4A', '5', '6', '7A', '8A', '9', '10']

img files   : 0
D files     : 0
FAT files   : 0

Global crops  : 0
Desc figures  : 12
Count match   : ✗ MISMATCH — all will be _Fu



""


In [3]:
# ─── Full run — match and rename all patents in raw/ ─────────────────────────
# Patents without a download folder are silently skipped.

raw_dir     = cfg["paths"]["raw_images"]
all_summary = []
run_errors  = []

for patent_id, meta in excel_index.items():
    img_dir = raw_dir / patent_id
    if not img_dir.exists():
        continue

    description_text = meta.get("description_of_drawings") or ""

    try:
        matches = match_positionally(patent_id, img_dir, description_text)
        if not matches:
            continue

        summary              = rename_matched_files(matches, img_dir)
        summary["patent_id"] = patent_id
        all_summary.append(summary)

        n_F  = summary["renamed_F"]
        n_Fu = summary["renamed_Fu"]
        icon = "✓" if n_Fu == 0 else "⚠"
        print(f"  {icon}  {patent_id:<30}  F={n_F:>4}  Fu={n_Fu:>3}")

    except Exception as exc:
        run_errors.append((patent_id, str(exc)))
        print(f"  ✗  {patent_id}: {exc}")

print(f"\nDone. Patents processed: {len(all_summary)}  Errors: {len(run_errors)}")


Done. Patents processed: 0  Errors: 0


In [4]:
# ─── Matching report ─────────────────────────────────────────────────────────
if not all_summary:
    print("No results yet — run Cell 3 first.")
else:
    import pandas as pd

    df = pd.DataFrame(all_summary)

    total_patents   = len(df)
    perfect         = int((df["renamed_Fu"] == 0).sum())
    needs_review    = int((df["renamed_Fu"] > 0).sum())
    total_F         = int(df["renamed_F"].sum())
    total_Fu        = int(df["renamed_Fu"].sum())
    total_figs      = total_F + total_Fu
    kept_originals  = int(df["kept_originals"].sum())

    def _pct(n):
        return f"{n / total_patents * 100:.0f}%" if total_patents else "N/A"

    print("══════════════════════════════════════")
    print("Figure Matching Report")
    print("══════════════════════════════════════")
    print(f"Patents processed       : {total_patents}")
    print(f"Perfect matches (_F)    : {perfect}  ({_pct(perfect)})")
    print(f"Needs review (_Fu)      : {needs_review}  ({_pct(needs_review)})")
    print(f"Total figures matched   : {total_figs}")
    print(f"  - labeled   (_F)      : {total_F}")
    print(f"  - unlabeled (_Fu)     : {total_Fu}")
    print(f"D/FAT originals kept    : {kept_originals}")
    print(f"Run errors              : {len(run_errors)}")
    print("══════════════════════════════════════")

    if run_errors:
        print("\nErrors:")
        for pid, err in run_errors:
            print(f"  {pid}: {err}")

No results yet — run Cell 3 first.
